# Parte 6 — Teste de estresse: mudança de escala

**Opção escolhida:** *"Mudança de escala: avaliem em 0,5× e 2×. Por que uma rede
totalmente convolucional não é invariante a escala, e o que o ASPP faz (ou não
faz) a respeito?"*

É a continuação natural da Parte 5: lá o diagnóstico foi que o gargalo do
modelo é a **resolução** (não o campo receptivo), e a correção foi subir a
entrada de 128 para 256. Aqui a pergunta é o inverso — o que acontece quando a
escala de **teste** difere da de **treino** sem retreinar — e se o *Atrous
Spatial Pyramid Pooling* (slides 39–42) muda alguma coisa.

## O que é comparado

| modelo | treino | avaliado em | por quê |
|---|---|---|---|
| **U-Net** (Trilha A, checkpoint da Parte 4) | 128×128 | 64 (0,5×), 128 (1×), 256 (2×) | o modelo final |
| **DeepLabV3** (Trilha A, treinada aqui com a mesma receita) | 128×128 | 64, 128, 256 | mesma perda, mesmas cabeças, mesmo split; só muda o backbone — *atrous* + ASPP em vez de 3 pools + skips. É o que isola "o que o ASPP faz" |
| **U-Net @256** (checkpoint da correção da Parte 5, se existir) | 256×256 | 128 (0,5×), 256 (1×), 512 (2×) | se a degradação acompanha a escala de **treino**, o problema é um *prior* de escala aprendido, não um defeito de uma resolução específica |

Em todas: o mesmo split de validação (`SEED=67`, 134 imagens), a imagem e o
GT reamostrados para a resolução de teste, e a decodificação por watershed com
`MIN_MARKER_SIZE` proporcional à escala (1 / 2 / 4 / 8 px em 64 / 128 / 256 /
512), para que o decodificador não seja a variável. As métricas são as das
Partes 1–5 (matching guloso por IoU decrescente; mAP@[.50:.95]; erro de
contagem; e a classificação de erros da Parte 5 — perdido / fundido / partido /
mal delimitado / falso positivo).

Além das métricas, medimos uma **assinatura direta da não-invariância**: a
espessura da faixa de *fronteira* que a rede prevê, em pixels, em cada escala.
Se a rede fosse invariante, a faixa cresceria com a imagem (2 px a 2×); se os
filtros têm extensão fixa em pixels, a faixa fica com ~1 px em qualquer escala.

In [1]:
# ============================================================
# IMPORTS
# ============================================================

import sys
import os

sys.path.append(os.path.abspath(".."))

import json
import copy
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch
import torch.nn as nn

from torch.utils.data import DataLoader, random_split
from scipy import ndimage

from src.dataset.bbbc038 import BBBC038Dataset, SegmentationTransform

from src.nn.models import TrilhaAModel
from src.nn.optimizers import create_optimizer

from src.nn.loss import focal_loss, l1_loss

from src.nn.targets import instance_map_to_boundary_targets, decode_watershed

from src.nn.metrics import (
    binary_iou_dice,
    label_map_iou_matrix,
    label_map_overlap_matrix,
    instance_scores_from_map,
    greedy_match_from_iou,
    count_tp_fp_fn_from_iou,
    mean_average_precision_from_iou,
)

from src.nn.tiling import compact_labels

In [2]:
# ============================================================
# CONFIGURACOES
# ============================================================

SEED = 67                      # mesmo split das Partes 1, 2, 4 e 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

DATA_ROOT = "../data/stage1_train"

# ------------------------------------------------------------
# Receita de treino (a mesma da Parte 4) -- usada so se faltar checkpoint
# ------------------------------------------------------------

MIN_INST_PIXELS = 4
LOSS_CFG        = dict(kind="focal", gamma=2.0, balanced=True)
DISTANCE_WEIGHT = 1.0
NUM_EPOCHS      = 10
BATCH_SIZE      = 16
INTERIOR_THRESH = 0.50

# espessura da fronteira no treino e tamanho minimo de marcador na
# decodificacao, proporcionais a resolucao (128 -> 1 px / 2 px, como na Parte 4)
BOUNDARY_WIDTH  = {64: 1, 128: 1, 256: 2, 512: 4}
MIN_MARKER_SIZE = {64: 1, 128: 2, 256: 4, 512: 8}

# ------------------------------------------------------------
# Modelos e escalas
# ------------------------------------------------------------

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

MODELS = {
    # nome: (backbone, resolucao de treino, checkpoint)
    "U-Net @128":     ("unet",    128, os.path.join(RESULTS_DIR, "4_trilha_a_unet.pt")),
    "DeepLabV3 @128": ("deeplab", 128, os.path.join(RESULTS_DIR, "6_trilha_a_deeplab.pt")),
    "U-Net @256":     ("unet",    256, os.path.join(RESULTS_DIR, "5_trilha_a_unet_ft256.pt")),
}
RELATIVE_SCALES = [0.5, 1.0, 2.0]

MODEL_COLORS = {"U-Net @128": "#4c72b0", "DeepLabV3 @128": "#c44e52", "U-Net @256": "#55a868"}
ERROR_KEYS = ("perdido", "fundido", "partido", "mal delimitado", "falso positivo")
ERROR_COLORS = {
    "ok": (0.35, 0.35, 0.35), "perdido": (1.0, 0.15, 0.15), "fundido": (1.0, 0.55, 0.0),
    "partido": (1.0, 1.0, 0.2), "mal delimitado": (0.7, 0.3, 0.9), "falso positivo": (0.2, 0.5, 1.0),
}

Device: cpu


## Alvos, treino, decodificação, dados e avaliação (os mesmos das Partes 4–5)

In [3]:
def build_targets(instance_map_batch, boundary_width):
    sem, dist = [], []
    for i in range(instance_map_batch.shape[0]):
        s, d = instance_map_to_boundary_targets(
            instance_map_batch[i].detach().cpu().numpy(),
            boundary_width=boundary_width, min_pixels=MIN_INST_PIXELS,
        )
        sem.append(s); dist.append(d)
    return torch.stack(sem), torch.stack(dist)


def train_epoch(dataloader, model, optimizer, boundary_width):
    model.train()
    acc = {"loss": 0.0, "semantic": 0.0, "distance": 0.0}
    for X, instance_map in dataloader:
        X = X.to(device)
        sem_t, dist_t = build_targets(instance_map, boundary_width)
        sem_t, dist_t = sem_t.to(device), dist_t.to(device)
        sem_logits, dist_logits = model(X)
        dist_pred = torch.sigmoid(dist_logits).squeeze(1)
        sem_loss = focal_loss(sem_logits, sem_t, class_weights="balanced", gamma=LOSS_CFG["gamma"])
        dist_loss = l1_loss(dist_pred, dist_t)
        loss = sem_loss + DISTANCE_WEIGHT * dist_loss
        optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
        acc["loss"] += loss.item(); acc["semantic"] += sem_loss.item(); acc["distance"] += dist_loss.item()
    n = len(dataloader)
    return {k: v / n for k, v in acc.items()}


def make_split(size, augment):
    transform = SegmentationTransform(size=(size, size), augment=augment)
    dataset = BBBC038Dataset(root=DATA_ROOT, transform=transform, mode="instance")
    n_total = len(dataset); n_train = int(0.8 * n_total); n_val = n_total - n_train
    return random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(SEED))


def load_or_train(backbone, checkpoint, size, epochs, tag):
    model = TrilhaAModel(backbone).to(device)
    if os.path.exists(checkpoint):
        ckpt = torch.load(checkpoint, map_location=device)
        model.load_state_dict(ckpt["state_dict"])
        print(f"[{tag}] checkpoint carregado: {checkpoint}")
        return model.eval()
    train_split, _ = make_split(size, augment=True)
    loader = DataLoader(train_split, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    optimizer = create_optimizer("Adam", model, lr=1e-3)
    history, t0 = [], time.time()
    for epoch in range(epochs):
        logs = train_epoch(loader, model, optimizer, BOUNDARY_WIDTH[size])
        history.append(logs)
        print(f"[{tag}] epoca {epoch + 1:2d}/{epochs} | loss {logs['loss']:.4f} | "
              f"semantic {logs['semantic']:.4f} | distance {logs['distance']:.4f} | {(time.time() - t0) / 60:.1f} min")
    config = {"backbone": backbone, "loss": LOSS_CFG, "boundary_width": BOUNDARY_WIDTH[size], "image_size": [size, size],
              "epochs": epochs, "min_marker_size": MIN_MARKER_SIZE[size], "split_seed": SEED}
    torch.save({"state_dict": model.state_dict(), "history": history, "config": config}, checkpoint)
    print(f"[{tag}] checkpoint salvo em {checkpoint}")
    return model.eval()


def watershed_from_outputs(sem_prob, dist_pred, min_marker_size):
    return decode_watershed(
        interior_mask=sem_prob[1] > INTERIOR_THRESH,
        foreground_mask=sem_prob.argmax(axis=0) != 0,
        landscape=-dist_pred, min_marker_size=min_marker_size,
    )


@torch.no_grad()
def forward_np(model, batch):
    model.eval()
    sem_logits, dist_logits = model(batch.to(device))
    return torch.softmax(sem_logits, dim=1).cpu().numpy(), torch.sigmoid(dist_logits).squeeze(1).cpu().numpy()


def classify_errors(pred, gt, iou, iou_threshold=0.5, cover=0.5, piece=0.2):
    # a mesma classificacao da Parte 5
    intersection, pred_area, gt_area = label_map_overlap_matrix(pred, gt)
    n_pred, n_gt = intersection.shape
    pairs = greedy_match_from_iou(iou, iou_threshold)
    matched_gt = {g for _, g in pairs}; matched_pred = {p for p, _ in pairs}
    categories = ["ok"] * n_gt; fp_flags = [False] * n_pred
    if n_gt > 0 and n_pred > 0:
        cover_gt = intersection / np.maximum(gt_area[None, :], 1)
        major = cover_gt.argmax(axis=0); major_cov = cover_gt.max(axis=0)
        shared = np.bincount(major[major_cov >= cover], minlength=n_pred)
        cover_pred = intersection / np.maximum(pred_area[:, None], 1)
        for g in range(n_gt):
            if g in matched_gt:
                continue
            if major_cov[g] >= cover and shared[major[g]] >= 2:
                categories[g] = "fundido"
            elif (cover_gt[:, g] >= piece).sum() >= 2:
                categories[g] = "partido"
            elif major_cov[g] < piece:
                categories[g] = "perdido"
            else:
                categories[g] = "mal delimitado"
        for p in range(n_pred):
            fp_flags[p] = (p not in matched_pred) and (cover_pred[p].max() < piece)
    elif n_gt > 0:
        categories = ["perdido"] * n_gt
    elif n_pred > 0:
        fp_flags = [True] * n_pred
    return categories, fp_flags


def boundary_thickness(sem_class):
    '''
    Espessura media (px) da faixa da classe fronteira: area / (perimetro / 2),
    com o perimetro contado em ARESTAS de pixel entre a faixa e o resto
    (4-vizinhanca). Uma casca fechada de largura w em torno de um nucleo de
    perimetro P tem area ~ w*P e perimetro ~ 2P -> espessura ~ w, inclusive
    para w = 1.
    '''
    band = sem_class == 2
    if band.sum() == 0:
        return np.nan
    pad = np.pad(band, 1)
    core = pad[1:-1, 1:-1]
    edges = ((core & ~pad[:-2, 1:-1]).sum() + (core & ~pad[2:, 1:-1]).sum()
             + (core & ~pad[1:-1, :-2]).sum() + (core & ~pad[1:-1, 2:]).sum())
    return float(2.0 * band.sum() / max(edges, 1))


@torch.no_grad()
def evaluate_split(model, size, batch_size=16):
    '''Avalia o modelo na validacao reamostrada para `size`. Devolve registros por imagem + resumo.'''
    _, val_split = make_split(size, augment=False)
    loader = DataLoader(val_split, batch_size=batch_size, shuffle=False, num_workers=0)
    mm = MIN_MARKER_SIZE[size]

    records, pos = [], 0
    for X, inst in loader:
        sem_prob, dist_pred = forward_np(model, X)
        for i in range(X.shape[0]):
            gt = compact_labels(inst[i].numpy())
            pred = compact_labels(watershed_from_outputs(sem_prob[i], dist_pred[i], mm))
            sem_class = sem_prob[i].argmax(axis=0)
            iou = label_map_iou_matrix(pred, gt)
            categories, fp_flags = classify_errors(pred, gt, iou)
            tp = len(greedy_match_from_iou(iou, 0.5)); n_pred, n_gt = iou.shape
            gt_sem, _ = instance_map_to_boundary_targets(gt, boundary_width=BOUNDARY_WIDTH[size], min_pixels=1)
            gt_sem = gt_sem.numpy()
            rec = {
                "pos": pos, "n_gt": n_gt, "n_pred": n_pred, "tp": tp, "fp": n_pred - tp, "fn": n_gt - tp,
                "f1": 2 * tp / max(2 * tp + (n_pred - tp) + (n_gt - tp), 1),
                "sem_iou": binary_iou_dice(pred > 0, gt > 0)[0],
                "iou": iou, "scores": instance_scores_from_map(sem_prob[i][1] + sem_prob[i][2], pred),
                "errors": {c: categories.count(c) for c in ERROR_KEYS if c != "falso positivo"},
                "thick_pred": boundary_thickness(sem_class), "thick_gt": boundary_thickness(gt_sem),
                "frac_boundary_pred": float((sem_class == 2).sum() / max((sem_class != 0).sum(), 1)),
                "frac_boundary_gt": float((gt_sem == 2).sum() / max((gt_sem != 0).sum(), 1)),
                "mean_area_pred": float(np.bincount(pred.ravel())[1:].mean()) if n_pred else np.nan,
                "mean_area_gt": float(np.bincount(gt.ravel())[1:].mean()) if n_gt else np.nan,
            }
            rec["errors"]["falso positivo"] = int(sum(fp_flags))
            records.append(rec); pos += 1

    map_value, aps = mean_average_precision_from_iou([r["iou"] for r in records], [r["scores"] for r in records])
    tp50, fp50, fn50 = count_tp_fp_fn_from_iou([r["iou"] for r in records], 0.5)
    summary = {
        "resolucao": size, "n_gt": int(sum(r["n_gt"] for r in records)),
        "IoU (semantico)": float(np.mean([r["sem_iou"] for r in records])),
        "mAP@[.50:.95]": float(map_value), "AP@0.50": float(aps[0.5]), "AP@0.75": float(aps[0.75]),
        "TP@0.50": int(tp50), "FP@0.50": int(fp50), "FN@0.50": int(fn50),
        "erro medio de contagem": float(np.mean([abs(r["n_pred"] - r["n_gt"]) for r in records])),
        "espessura fronteira prevista (px)": float(np.nanmean([r["thick_pred"] for r in records])),
        "espessura fronteira GT (px)": float(np.nanmean([r["thick_gt"] for r in records])),
        "fracao fronteira prevista": float(np.mean([r["frac_boundary_pred"] for r in records])),
        "fracao fronteira GT": float(np.mean([r["frac_boundary_gt"] for r in records])),
        "area media prevista / GT": float(np.nanmean([r["mean_area_pred"] / r["mean_area_gt"] for r in records
                                                      if r["n_pred"] and r["n_gt"]])),
        **{c: int(sum(r["errors"][c] for r in records)) for c in ERROR_KEYS},
    }
    return records, summary

## Modelos

A U-Net @128 é o checkpoint da Parte 4; a U-Net @256 é o da correção da Parte 5
(se o arquivo não existir, esse modelo é pulado). A **DeepLabV3** é treinada
aqui, do zero, com exatamente a receita da Parte 4 (mesma perda, mesmas cabeças,
mesmo split, 10 épocas a 128×128) e salva em `results/6_trilha_a_deeplab.pt` —
com o checkpoint presente a célula só carrega.

In [ ]:
models = {}
for name, (backbone, train_size, checkpoint) in MODELS.items():
    if name == "U-Net @256" and not os.path.exists(checkpoint):
        print(f"[{name}] checkpoint {checkpoint} nao encontrado -- rode o 5_failure_gallery.ipynb; modelo pulado")
        continue
    models[name] = load_or_train(backbone, checkpoint, train_size, NUM_EPOCHS, tag=name)

for name, m in models.items():
    print(f"{name:15s}: {sum(p.numel() for p in m.parameters()):,} parametros")

## Avaliação a 0,5×, 1× e 2× da escala de treino de cada modelo

In [ ]:
# ============================================================
# AVALIACAO  --  cada modelo em 0.5x / 1x / 2x da SUA escala de treino
# ============================================================

results = {}            # (modelo, escala relativa) -> (records, summary)
t_all = time.time()

for name, m in models.items():
    train_size = MODELS[name][1]
    for rel in RELATIVE_SCALES:
        size = int(train_size * rel)
        t0 = time.time()
        recs, summ = evaluate_split(m, size)
        results[(name, rel)] = (recs, summ)
        print(f"{name:15s} | {rel:>3}x = {size:3d}px | mAP {summ['mAP@[.50:.95]']:.4f} | AP50 {summ['AP@0.50']:.4f} | "
              f"n_gt {summ['n_gt']} | erro contagem {summ['erro medio de contagem']:.1f} | {time.time() - t0:.0f}s")

# imagem "tipica" para a figura qualitativa: F1 mediano a 1x (U-Net @128) entre as que tem >= 20 nucleos
cands = sorted([r for r in results[("U-Net @128", 1.0)][0] if r["n_gt"] >= 20], key=lambda r: r["f1"])
SHOWCASE_POS = cands[len(cands) // 2]["pos"]

print(f"\ntotal {(time.time() - t_all) / 60:.1f} min | imagem de demonstracao: #{SHOWCASE_POS}")

In [ ]:
# ============================================================
# TABELA COMPLETA
# ============================================================

table = pd.DataFrame({f"{name} | {rel}x ({summ['resolucao']}px)": summ
                      for (name, rel), (_, summ) in results.items()})
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
pd.set_option("display.width", 250)
table

In [ ]:
# ============================================================
# CURVAS DE DEGRADACAO
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
metrics = [("mAP@[.50:.95]", "mAP@[.50:.95]"), ("AP@0.50", "AP@0.50"), ("erro medio de contagem", "erro medio de contagem")]

for ax, (key, label) in zip(axes, metrics):
    for name in models:
        ys = [results[(name, rel)][1][key] for rel in RELATIVE_SCALES]
        ax.plot(RELATIVE_SCALES, ys, marker="o", label=name, color=MODEL_COLORS[name])
    ax.set_xscale("log", base=2); ax.set_xticks(RELATIVE_SCALES); ax.set_xticklabels([f"{s}x" for s in RELATIVE_SCALES])
    ax.set_xlabel("escala de teste / escala de treino"); ax.set_ylabel(label); ax.grid(alpha=0.3); ax.legend()
plt.suptitle("Degradacao com a escala -- cada modelo relativo a sua propria resolucao de treino")
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# QUE ERRO APARECE EM CADA ESCALA?  (U-Net @128 vs DeepLabV3 @128)
# ============================================================

pair = [n for n in ("U-Net @128", "DeepLabV3 @128") if n in models]
fig, axes = plt.subplots(1, len(pair), figsize=(8 * len(pair), 5), squeeze=False)
x = np.arange(len(ERROR_KEYS)); w = 0.25

for ax, name in zip(axes[0], pair):
    for k, rel in enumerate(RELATIVE_SCALES):
        summ = results[(name, rel)][1]
        ax.bar(x + (k - 1) * w, [summ[c] for c in ERROR_KEYS], width=w, label=f"{rel}x ({summ['resolucao']}px), n_gt={summ['n_gt']}")
    ax.set_xticks(x); ax.set_xticklabels(ERROR_KEYS, rotation=15); ax.set_ylabel("nucleos (validacao)")
    ax.set_title(name); ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

## Assinatura da não-invariância: a faixa de fronteira tem espessura fixa em pixels

O alvo de treino é uma casca de fronteira com espessura proporcional à
resolução (1 px a 128, 2 px a 256). Se a rede fosse invariante a escala, a
casca **prevista** a 2× teria o dobro da espessura da casca a 1×; se os filtros
têm extensão fixa em pixels, ela fica praticamente igual. A tabela abaixo mede
isso (espessura ≈ área / meio-perímetro da faixa prevista), ao lado da fração
do foreground previsto que é fronteira e da razão entre a área média das
instâncias previstas e a das verdadeiras.

In [ ]:
rows = []
for (name, rel), (_, summ) in results.items():
    rows.append({"modelo": name, "escala": f"{rel}x", "px": summ["resolucao"],
                 "espessura prevista (px)": summ["espessura fronteira prevista (px)"],
                 "espessura do alvo (px)": summ["espessura fronteira GT (px)"],
                 "fracao fronteira prevista": summ["fracao fronteira prevista"],
                 "fracao fronteira do alvo": summ["fracao fronteira GT"],
                 "area prevista / GT": summ["area media prevista / GT"]})
thickness = pd.DataFrame(rows).set_index(["modelo", "escala"])
thickness

In [ ]:
# ============================================================
# FIGURA QUALITATIVA  --  a mesma imagem nas tres escalas (U-Net @128 e DeepLabV3 @128)
# ============================================================

def colorize(label_map, seed=0):
    n = int(label_map.max()); rng = np.random.default_rng(seed)
    lut = rng.uniform(0.25, 1.0, size=(n + 1, 3)).astype(np.float32); lut[0] = 0.0
    return lut[label_map]


def error_panel(pred, gt, categories, fp_flags):
    rgb = np.zeros(gt.shape + (3,), dtype=np.float32)
    for g, cat in enumerate(categories, start=1):
        rgb[gt == g] = ERROR_COLORS[cat]
    for p, flag in enumerate(fp_flags, start=1):
        if flag:
            rgb[(pred == p) & (gt == 0)] = ERROR_COLORS["falso positivo"]
    return rgb


# reavalia so a imagem de demonstracao com saidas densas em todas as escalas
dense = {}
for name in pair:
    for rel in RELATIVE_SCALES:
        size = int(MODELS[name][1] * rel)
        _, val_split = make_split(size, augment=False)
        X, inst = val_split[SHOWCASE_POS]
        sem_prob, dist_pred = forward_np(models[name], X[None])
        gt = compact_labels(inst.numpy())
        pred = compact_labels(watershed_from_outputs(sem_prob[0], dist_pred[0], MIN_MARKER_SIZE[size]))
        iou = label_map_iou_matrix(pred, gt)
        cats, fps = classify_errors(pred, gt, iou)
        f1 = results[(name, rel)][0][SHOWCASE_POS]["f1"]
        dense[(name, rel)] = dict(image=X.numpy(), gt=gt, pred=pred, sem_class=sem_prob[0].argmax(axis=0),
                                  dist=dist_pred[0], cats=cats, fps=fps, f1=f1, size=size)

fig, ax = plt.subplots(len(RELATIVE_SCALES), 7, figsize=(24, 3.6 * len(RELATIVE_SCALES)))
for r, rel in enumerate(RELATIVE_SCALES):
    u = dense[("U-Net @128", rel)]
    ax[r, 0].imshow(np.transpose(u["image"], (1, 2, 0))); ax[r, 0].set_title(f"{rel}x = {u['size']}px", fontsize=10)
    ax[r, 1].imshow(colorize(u["gt"])); ax[r, 1].set_title(f"GT ({int(u['gt'].max())})", fontsize=10)
    ax[r, 2].imshow(u["sem_class"], vmin=0, vmax=2, cmap="viridis"); ax[r, 2].set_title("U-Net: argmax 3 classes", fontsize=10)
    ax[r, 3].imshow(u["dist"], vmin=0, vmax=1, cmap="magma"); ax[r, 3].set_title("U-Net: distancia", fontsize=10)
    ax[r, 4].imshow(error_panel(u["pred"], u["gt"], u["cats"], u["fps"])); ax[r, 4].set_title(f"U-Net: erros (F1 {u['f1']:.2f})", fontsize=10)
    if ("DeepLabV3 @128", rel) in dense:
        d = dense[("DeepLabV3 @128", rel)]
        ax[r, 5].imshow(d["sem_class"], vmin=0, vmax=2, cmap="viridis"); ax[r, 5].set_title("DeepLab: argmax 3 classes", fontsize=10)
        ax[r, 6].imshow(error_panel(d["pred"], d["gt"], d["cats"], d["fps"])); ax[r, 6].set_title(f"DeepLab: erros (F1 {d['f1']:.2f})", fontsize=10)
    for a in ax[r]:
        a.axis("off")
ax[0, 6].legend(handles=[patches.Patch(color=c, label=n) for n, c in ERROR_COLORS.items()],
                loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=8, frameon=False)
plt.suptitle(f"Imagem #{SHOWCASE_POS} da validacao nas tres escalas (modelos treinados a 128px)", fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# SALVA OS NUMEROS DA PARTE 6
# ============================================================

payload = {
    "config": {"scales": RELATIVE_SCALES, "models": {k: {"backbone": v[0], "train_size": v[1], "checkpoint": v[2]}
                                                      for k, v in MODELS.items() if k in models},
               "min_marker_size": MIN_MARKER_SIZE, "boundary_width": BOUNDARY_WIDTH, "showcase_pos": SHOWCASE_POS},
    "resultados": {f"{name} | {rel}x": summ for (name, rel), (_, summ) in results.items()},
}
with open(os.path.join(RESULTS_DIR, "6_stress_test.json"), "w") as fh:
    json.dump(payload, fh, indent=2)

print(json.dumps({k: {m: v[m] for m in ("resolucao", "n_gt", "mAP@[.50:.95]", "AP@0.50", "erro medio de contagem",
                                          "espessura fronteira prevista (px)", "espessura fronteira GT (px)")}
                  for k, v in payload["resultados"].items()}, indent=2))

## Por que uma rede totalmente convolucional não é invariante a escala — e o que o ASPP faz (ou não)

*(preenchido depois de rodar — ver célula final)*